In [ ]:
# --- 1. IMPORT & SETUP ---
import os
import glob
import random
import torch
import numpy as np
import cv2
import mediapipe as mp 
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg19
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import save_image
import time
import datetime
import sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# PATHS
DATASET_ROOT = r"D:\senior_project\dataset\CelebAMask-HQ"
IMG_DIR = os.path.join(DATASET_ROOT, "CelebA-HQ-img")
MASK_DIR = os.path.join(DATASET_ROOT, "CelebAMask-HQ-mask-anno")


EXPERIMENT_NAME = "result_v9_5channels" 
OUTPUT_IMAGE_DIR = os.path.join(EXPERIMENT_NAME, "images")
OUTPUT_MODEL_DIR = os.path.join(EXPERIMENT_NAME, "models")

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)


# --- 2. DATASET ---
class InpaintingDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, mode="train"):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.mode = mode
        
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5
        )
        
        self.mask_paths = []
        for i in range(15):
            self.mask_paths.extend(glob.glob(os.path.join(mask_dir, str(i), "*_eye_g.png")))
        print(f"โหลด Mask แว่นตา: {len(self.mask_paths)} รูป")
        
        self.all_images = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        
    def __len__(self):
        return len(self.all_images)

    def get_landmarks_image(self, image_np, h, w):
        landmark_map = np.zeros((h, w), dtype=np.uint8)
        results = self.face_mesh.process(image_np)
        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0].landmark
            def to_px(idx): return (int(landmarks[idx].x * w), int(landmarks[idx].y * h))
            LEFT = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
            RIGHT = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
            pts_l = np.array([to_px(i) for i in LEFT], np.int32).reshape((-1, 1, 2))
            cv2.polylines(landmark_map, [pts_l], True, 255, 2)
            pts_r = np.array([to_px(i) for i in RIGHT], np.int32).reshape((-1, 1, 2))
            cv2.polylines(landmark_map, [pts_r], True, 255, 2)
        return Image.fromarray(landmark_map)

    def __getitem__(self, idx):
        img_path = self.all_images[idx]
        image = Image.open(img_path).convert("RGB")
        
        # กรองรูปที่มีแว่นจริงออก (ใช้หน้าสดเท่านั้น)
        try:
            img_id = int(os.path.basename(img_path).split('.')[0])
            real_mask = os.path.join(self.mask_dir, str(img_id // 2000), f"{img_id:05d}_eye_g.png")
            if os.path.exists(real_mask): return self.__getitem__(random.randint(0, len(self.all_images)-1))
        except: pass

        # สุ่ม Mask
        mask = Image.open(random.choice(self.mask_paths)).convert("L")
        img_np = np.array(image)
        landmark_img = self.get_landmarks_image(img_np, img_np.shape[0], img_np.shape[1])

        # Augmentation
        if self.mode == "train":
            if random.random() > 0.5:
                image = transforms.functional.hflip(image)
                landmark_img = transforms.functional.hflip(landmark_img)
            if random.random() > 0.5:
                angle = random.randint(-20, 20)
                image = transforms.functional.rotate(image, angle)
                mask = transforms.functional.rotate(mask, angle)
                landmark_img = transforms.functional.rotate(landmark_img, angle)

        if self.transform: image = self.transform(image)
            
        resize = transforms.Resize((256, 256))
        mask = resize(mask)
        landmark_img = resize(landmark_img)
        
        mask_tensor = transforms.ToTensor()(mask)          # Channel 4 (Mask)
        landmark_tensor = transforms.ToTensor()(landmark_img) # Channel 5 (Landmark)
        
        masked_image = image * (1 - mask_tensor)           # Channel 1-3 (RGB with Hole)
        
        #รวม 5 Channels: [RGB(3) + Mask(1) + Landmark(1)]
        model_input = torch.cat((masked_image, mask_tensor, landmark_tensor), 0)
        
        return model_input, image, mask_tensor

# --- 3. MODELS (Generator รับ 5 Channels) ---
class UNetDown(nn.Module):
    def __init__(self, in_size, out_size, normalize=True, dropout=0.0):
        super(UNetDown, self).__init__()
        layers = [nn.Conv2d(in_size, out_size, 4, 2, 1, bias=False)]
        if normalize: layers.append(nn.InstanceNorm2d(out_size))
        layers.append(nn.LeakyReLU(0.2))
        if dropout: layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_size, out_size, dropout=0.0):
        super(UNetUp, self).__init__()
        layers = [
            nn.ConvTranspose2d(in_size, out_size, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_size),
            nn.ReLU(inplace=True)
        ]
        if dropout: layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.model(x)
        x = torch.cat((x, skip), 1)
        return x

class GeneratorUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=3):
        super(GeneratorUNet, self).__init__()
        # Encoder
        self.down1 = UNetDown(in_channels, 64, normalize=False)
        self.down2 = UNetDown(64, 128)
        self.down3 = UNetDown(128, 256)
        self.down4 = UNetDown(256, 512, dropout=0.5)
        self.down5 = UNetDown(512, 512, dropout=0.5)
        self.down6 = UNetDown(512, 512, dropout=0.5)
        self.down7 = UNetDown(512, 512, dropout=0.5)
        self.down8 = UNetDown(512, 512, normalize=False, dropout=0.5)
        # Decoder
        self.up1 = UNetUp(512, 512, dropout=0.5)
        self.up2 = UNetUp(1024, 512, dropout=0.5)
        self.up3 = UNetUp(1024, 512, dropout=0.5)
        self.up4 = UNetUp(1024, 512, dropout=0.5)
        self.up5 = UNetUp(1024, 256)
        self.up6 = UNetUp(512, 128)
        self.up7 = UNetUp(256, 64)
        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(128, out_channels, 4, padding=1),
            nn.Tanh()
        )
    def forward(self, x):
        d1 = self.down1(x); d2 = self.down2(d1); d3 = self.down3(d2); d4 = self.down4(d3)
        d5 = self.down5(d4); d6 = self.down6(d5); d7 = self.down7(d6); d8 = self.down8(d7)
        u1 = self.up1(d8, d7); u2 = self.up2(u1, d6); u3 = self.up3(u2, d5); u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3); u6 = self.up6(u5, d2); u7 = self.up7(u6, d1)
        return self.final(u7)

class Discriminator(nn.Module): # Discriminator 
    def __init__(self, in_channels=3, input_shape=(3, 256, 256)):
        super(Discriminator, self).__init__()
        def d_block(in_f, out_f, norm=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride=2, padding=1)]
            if norm: layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *d_block(in_channels, 64, False),
            *d_block(64, 128),
            *d_block(128, 256),
            *d_block(256, 512),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(512, 1, 4, padding=1, bias=False)
        )
    def forward(self, img): return self.model(img)

class VGG19(torch.nn.Module):
    def __init__(self):
        super(VGG19, self).__init__()
        features = vgg19(weights='DEFAULT').features
        self.slice1 = torch.nn.Sequential(); self.slice2 = torch.nn.Sequential(); self.slice3 = torch.nn.Sequential()
        for x in range(2): self.slice1.add_module(str(x), features[x])
        for x in range(2, 7): self.slice2.add_module(str(x), features[x])
        for x in range(7, 12): self.slice3.add_module(str(x), features[x])
        for param in self.parameters(): param.requires_grad = False
    def forward(self, x):
        h1 = self.slice1(x); h2 = self.slice2(h1); h3 = self.slice3(h2)
        return [h1, h2, h3]

# --- 4. INIT & HELPERS ---
# สร้าง Generator แบบ 5 Channels
generator = GeneratorUNet(in_channels=5).to(device)
discriminator = Discriminator().to(device)
local_discriminator = Discriminator(input_shape=(3, 128, 128)).to(device)
vgg = VGG19().to(device); vgg.eval()

criterion_GAN = torch.nn.MSELoss()
criterion_pixel = torch.nn.L1Loss()
criterion_content = torch.nn.L1Loss()

optimizer_G = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_LD = torch.optim.Adam(local_discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1: torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm2d") != -1: torch.nn.init.normal_(m.weight.data, 1.0, 0.02); torch.nn.init.constant_(m.bias.data, 0.0)

def gram_matrix(input):
    features = input.view(input.size(0) * input.size(1), input.size(2) * input.size(3))
    G = torch.mm(features, features.t())
    return G.div(input.size(0) * input.size(1) * input.size(2) * input.size(3))

def prep_for_vgg(x): 
    vgg_mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
    vgg_std = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)
    return ((x + 1) / 2.0 - vgg_mean) / vgg_std 

def crop_eyes_region(imgs, masks, size=128):
    batch_size = imgs.shape[0]
    cropped_batch = []
    for i in range(batch_size):
        y_indices, x_indices = torch.where(masks[i, 0] > 0)
        if len(y_indices) > 0:
            y_min, y_max = y_indices.min(), y_indices.max()
            x_min, x_max = x_indices.min(), x_indices.max()
            cy = (y_min + y_max) // 2; cx = (x_min + x_max) // 2
            side = max(y_max - y_min, x_max - x_min) + 40
            half = side // 2
            y1 = max(0, int(cy - half)); y2 = min(imgs.shape[2], int(cy + half))
            x1 = max(0, int(cx - half)); x2 = min(imgs.shape[3], int(cx + half))
            roi = imgs[i, :, y1:y2, x1:x2]
        else:
            h, w = imgs.shape[2:]
            roi = imgs[i, :, h//4:3*h//4, w//4:3*w//4]
        roi_resized = F.interpolate(roi.unsqueeze(0), size=(size, size), mode='bilinear', align_corners=False)
        cropped_batch.append(roi_resized)
    return torch.cat(cropped_batch, dim=0)

# --- 5. TRAINING LOOP ---

# Initialize Weights (เริ่มใหม่หมด)
#generator.apply(weights_init_normal)
#discriminator.apply(weights_init_normal)
#local_discriminator.apply(weights_init_normal)

# เช็คว่ามีไฟล์เก่าไหม ถ้ามีให้โหลดมาเทรนต่อ
latest_gen = os.path.join(OUTPUT_MODEL_DIR, "generator_latest.pth")
latest_disc = os.path.join(OUTPUT_MODEL_DIR, "discriminator_latest.pth")
latest_ld = os.path.join(OUTPUT_MODEL_DIR, "local_discriminator_latest.pth")

if os.path.exists(latest_gen):
    print("พบไฟล์โมเดลเก่า กำลังโหลดเพื่อเทรนต่อ")
    generator.load_state_dict(torch.load(latest_gen))
    discriminator.load_state_dict(torch.load(latest_disc))
    local_discriminator.load_state_dict(torch.load(latest_ld))
else:
    print("ไม่พบไฟล์โมเดลเก่า เริ่มต้นเทรนใหม่จากศูนย์")
    generator.apply(weights_init_normal)
    discriminator.apply(weights_init_normal)
    local_discriminator.apply(weights_init_normal)

EPOCHS = 20
BATCH_SIZE = 4
SAMPLE_INTERVAL = 1000

gan_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = InpaintingDataset(IMG_DIR, MASK_DIR, transform=gan_transforms)
dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

def save_sample(batches_done):
    model_inputs, real_imgs, masks = next(iter(dataloader))
    model_inputs = model_inputs.to(device)
    real_imgs = real_imgs.to(device)
    masks = masks.to(device)
    with torch.no_grad(): output = generator(model_inputs)
    final_output = real_imgs * (1 - masks) + output * masks
    # โชว์ 3 Channel แรกของ Input (คือรูปหน้าเจาะรู)
    masked_input_vis = model_inputs[:, :3, :, :] 
    sample_img = torch.cat((masked_input_vis.data, final_output.data, real_imgs.data), -2)
    save_image(sample_img, os.path.join(OUTPUT_IMAGE_DIR, f"result_{batches_done}.png"), nrow=5, normalize=True)

# ลด LR ลงเมื่อถึงที่ต้องการ
scheduler_G = torch.optim.lr_scheduler.MultiStepLR(optimizer_G, milestones=[5], gamma=0.1)
scheduler_D = torch.optim.lr_scheduler.MultiStepLR(optimizer_D, milestones=[5], gamma=0.1)
scheduler_LD = torch.optim.lr_scheduler.MultiStepLR(optimizer_LD, milestones=[5], gamma=0.1)

print(f"\n--- เริ่มเทรน {EXPERIMENT_NAME} ---")
prev_time = time.time()

for epoch in range(EPOCHS):
    for i, (model_inputs, real_imgs, masks) in enumerate(dataloader):
        model_inputs = model_inputs.to(device)
        real_imgs = real_imgs.to(device)
        masks = masks.to(device)

        valid_global = torch.ones((model_inputs.size(0), 1, 16, 16), requires_grad=False).to(device)
        fake_global = torch.zeros((model_inputs.size(0), 1, 16, 16), requires_grad=False).to(device)
        valid_local = torch.ones((model_inputs.size(0), 1, 8, 8), requires_grad=False).to(device)
        fake_local = torch.zeros((model_inputs.size(0), 1, 8, 8), requires_grad=False).to(device)

        # Train Generator
        optimizer_G.zero_grad()
        gen_parts = generator(model_inputs)
        gen_imgs = real_imgs * (1 - masks) + gen_parts * masks
        gen_loc = crop_eyes_region(gen_imgs, masks, size=128)

        g_adv_global = criterion_GAN(discriminator(gen_imgs), valid_global)
        g_adv_local  = criterion_GAN(local_discriminator(gen_loc), valid_local)
        g_pixel = criterion_pixel(gen_parts * masks, real_imgs * masks)
        
        gen_feats = vgg(prep_for_vgg(gen_imgs))
        real_feats = vgg(prep_for_vgg(real_imgs))
        g_content = 0; g_style = 0
        for gf, rf in zip(gen_feats, real_feats):
            g_content += criterion_content(gf, rf)
            g_style += criterion_content(gram_matrix(gf), gram_matrix(rf))

        # สูตร Loss
        w_pixel = 10.0   
        w_local = 10.0
        g_style_weight = 100.0
        
        g_loss = (1 * g_adv_global) + \
                (w_local * g_adv_local) + \
                (w_pixel * g_pixel) + \
                (10 * g_content) + \
                (g_style_weight * g_style)

        g_loss.backward()
        optimizer_G.step()

        # Train Discriminators
        optimizer_D.zero_grad()
        d_loss_g = 0.5 * (criterion_GAN(discriminator(real_imgs), valid_global) + \
                          criterion_GAN(discriminator(gen_imgs.detach()), fake_global))
        d_loss_g.backward()
        optimizer_D.step()

        optimizer_LD.zero_grad()
        real_loc = crop_eyes_region(real_imgs, masks, size=128)
        gen_loc_d = crop_eyes_region(gen_imgs.detach(), masks, size=128)
        d_loss_l = 0.5 * (criterion_GAN(local_discriminator(real_loc), valid_local) + \
                          criterion_GAN(local_discriminator(gen_loc_d), fake_local))
        d_loss_l.backward()
        optimizer_LD.step()
        
        d_loss = d_loss_g + d_loss_l

        batches_done = epoch * len(dataloader) + i
        batches_left = EPOCHS * len(dataloader) - batches_done
        time_left = datetime.timedelta(seconds=batches_left * (time.time() - prev_time))
        prev_time = time.time()

        if i % 10 == 0:
            sys.stdout.write(
                f"\r[Epoch {epoch}/{EPOCHS}] [Batch {i}/{len(dataloader)}] "
                f"[D: {d_loss.item():.4f}] [G: {g_loss.item():.4f}] "
                f"ETA: {time_left} "
            )

        if batches_done % SAMPLE_INTERVAL == 0:
            save_sample(batches_done)

    # Save
    torch.save(generator.state_dict(), os.path.join(OUTPUT_MODEL_DIR, "generator_latest.pth"))
    torch.save(discriminator.state_dict(), os.path.join(OUTPUT_MODEL_DIR, "discriminator_latest.pth"))
    torch.save(local_discriminator.state_dict(), os.path.join(OUTPUT_MODEL_DIR, "local_discriminator_latest.pth"))
    
    if (epoch + 1) % 5 == 0:
         torch.save(generator.state_dict(), os.path.join(OUTPUT_MODEL_DIR, f"gen_epoch_{epoch+1}.pth"))

    scheduler_G.step()
    scheduler_D.step()
    scheduler_LD.step()    
    print(f"\n จบ Epoch {epoch}")

Using device: cuda
✅ Setup V9 เรียบร้อย (เตรียมเริ่มจากศูนย์)
🔄 พบไฟล์โมเดลเก่า! กำลังโหลดเพื่อเทรนต่อ...


C:\Users\Asus\AppData\Local\Temp\ipykernel_10872\3640160197.py:281: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  generator.load_state_dict(torch.load(latest_gen))
C:\Users\

✅ โหลด Mask แว่นตา: 1549 รูป

--- เริ่มเทรน result_v9_5channels (5 Channels: Fresh Start) ---
[Epoch 0/20] [Batch 7490/7500] [D: 0.3939] [G: 6.4051] ETA: 17:08:08.766723  61 
✅ จบ Epoch 0
[Epoch 1/20] [Batch 7490/7500] [D: 0.2110] [G: 5.2688] ETA: 14:44:57.798989  6 
✅ จบ Epoch 1
[Epoch 2/20] [Batch 7490/7500] [D: 0.2550] [G: 6.4246] ETA: 14:23:27.078452  7 
✅ จบ Epoch 2
[Epoch 3/20] [Batch 7490/7500] [D: 0.3367] [G: 6.2984] ETA: 13:09:17.766953  
✅ จบ Epoch 3
[Epoch 4/20] [Batch 7490/7500] [D: 0.3016] [G: 4.7375] ETA: 12:42:27.799754  
✅ จบ Epoch 4
[Epoch 5/20] [Batch 7490/7500] [D: 0.1910] [G: 6.9756] ETA: 11:59:56.791086  5 
✅ จบ Epoch 5
[Epoch 6/20] [Batch 7490/7500] [D: 0.2455] [G: 4.7815] ETA: 10:19:47.558029  
✅ จบ Epoch 6
[Epoch 7/20] [Batch 7490/7500] [D: 0.2211] [G: 5.7407] ETA: 9:11:43.724895  
✅ จบ Epoch 7
[Epoch 8/20] [Batch 7490/7500] [D: 0.2589] [G: 5.2490] ETA: 9:54:13.185582  
✅ จบ Epoch 8
[Epoch 9/20] [Batch 7490/7500] [D: 0.2345] [G: 6.3697] ETA: 8:32:26.434298  
✅ จ